In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries loaded!")

Libraries loaded!


In [3]:
# Business configuration
START_DATE = datetime(2021, 1, 1)
END_DATE = datetime(2026, 1, 31)
INITIAL_CUSTOMERS = 50

# Plan types and their monthly prices
PLANS = {
    'Basic':       29,
    'Professional': 99,
    'Enterprise':  299
}

# Plan distribution (what % of customers pick each)
PLAN_WEIGHTS = [0.5, 0.35, 0.15]

# Regions
REGIONS = ['North America', 'Europe', 'Asia Pacific', 
           'Latin America', 'Middle East']
REGION_WEIGHTS = [0.40, 0.25, 0.20, 0.10, 0.05]

print("Business parameters defined!")
print(f"Date range: {START_DATE.date()} to {END_DATE.date()}")
print(f"Plans: {PLANS}")

Business parameters defined!
Date range: 2021-01-01 to 2026-01-31
Plans: {'Basic': 29, 'Professional': 99, 'Enterprise': 299}


In [4]:
# Generate list of all customers
customers = []
customer_id = 1

# Starting customers on day 1
current_date = START_DATE

for month_offset in range(48):  # 48 months = 4 years
    current_date = START_DATE + timedelta(days=month_offset * 30)
    
    # New customers grow over time (business is growing)
    growth_factor = 1 + (month_offset * 0.05)
    new_customers = int(np.random.poisson(INITIAL_CUSTOMERS * growth_factor * 0.1))
    
    for _ in range(new_customers):
        plan = random.choices(list(PLANS.keys()), weights=PLAN_WEIGHTS)[0]
        region = random.choices(REGIONS, weights=REGION_WEIGHTS)[0]
        
        # Tenure in months before churning (longer for expensive plans)
        if plan == 'Enterprise':
            tenure = int(np.random.exponential(24))
        elif plan == 'Professional':
            tenure = int(np.random.exponential(18))
        else:
            tenure = int(np.random.exponential(12))
        
        tenure = max(1, min(tenure, 48))
        
        churn_date = current_date + timedelta(days=tenure * 30)
        if churn_date > END_DATE:
            churn_date = None  # Still active
            
        customers.append({
            'customer_id': f'CUST{customer_id:05d}',
            'signup_date': current_date,
            'plan': plan,
            'monthly_price': PLANS[plan],
            'region': region,
            'churn_date': churn_date,
            'tenure_months': tenure
        })
        customer_id += 1

df_customers = pd.DataFrame(customers)
print(f"Total customers generated: {len(df_customers)}")
print(f"\nPlan distribution:\n{df_customers['plan'].value_counts()}")
print(f"\nRegion distribution:\n{df_customers['region'].value_counts()}")

Total customers generated: 515

Plan distribution:
plan
Basic           252
Professional    185
Enterprise       78
Name: count, dtype: int64

Region distribution:
region
North America    198
Europe           137
Asia Pacific      91
Latin America     63
Middle East       26
Name: count, dtype: int64


In [5]:
# Build monthly snapshot of active customers and revenue
monthly_data = []

for month_offset in range(48):
    snapshot_date = START_DATE + timedelta(days=month_offset * 30)
    
    # Active customers this month
    active = df_customers[
        (df_customers['signup_date'] <= snapshot_date) &
        ((df_customers['churn_date'].isna()) | 
         (df_customers['churn_date'] > snapshot_date))
    ]
    
    # New customers this month
    month_start = snapshot_date
    month_end = snapshot_date + timedelta(days=30)
    new_custs = df_customers[
        (df_customers['signup_date'] >= month_start) &
        (df_customers['signup_date'] < month_end)
    ]
    
    # Churned customers this month
    churned = df_customers[
        (df_customers['churn_date'] >= month_start) &
        (df_customers['churn_date'] < month_end)
    ]
    
    # Calculate MRR
    mrr = active['monthly_price'].sum()
    
    # Churn rate
    churn_rate = len(churned) / len(active) * 100 if len(active) > 0 else 0
    
    monthly_data.append({
        'month': snapshot_date.strftime('%Y-%m'),
        'active_customers': len(active),
        'new_customers': len(new_custs),
        'churned_customers': len(churned),
        'mrr': mrr,
        'churn_rate': round(churn_rate, 2)
    })

df_monthly = pd.DataFrame(monthly_data)
print(f"Monthly data shape: {df_monthly.shape}")
print(f"\nFirst few months:")
print(df_monthly.head())
print(f"\nLatest MRR: ${df_monthly['mrr'].iloc[-1]:,.0f}")
print(f"Peak customers: {df_monthly['active_customers'].max():,}")

Monthly data shape: (48, 6)

First few months:
     month  active_customers  new_customers  churned_customers   mrr  \
0  2021-01                 5              5                  0   555   
1  2021-01                 8              5                  2   712   
2  2021-03                13              5                  0  1267   
3  2021-04                17              7                  3  1793   
4  2021-05                24              9                  2  2336   

   churn_rate  
0        0.00  
1       25.00  
2        0.00  
3       17.65  
4        8.33  

Latest MRR: $21,475
Peak customers: 185


In [6]:
# Save both datasets
df_customers.to_csv('../data/saas_customers.csv', index=False)
df_monthly.to_csv('../data/saas_monthly.csv', index=False)

print(f"Customers saved: {len(df_customers)} rows")
print(f"Monthly data saved: {len(df_monthly)} rows")
print("\nData generation complete!")

Customers saved: 515 rows
Monthly data saved: 48 rows

Data generation complete!


## Data Generation Summary

- Generated realistic SaaS business data (2021-2026)
- Total customers: 515 across 3 plan tiers
- Plan distribution: Basic 49%, Professional 36%, Enterprise 15%
- Regional distribution: North America leads at 38%
- Monthly snapshots: 48 months of MRR and churn data
- Latest MRR: $21,475 | Peak active customers: 185
- Churn modeled realistically — Enterprise churns slowest
- Seed set to 42 for full reproducibility